# **Mineral Prospectivity Project**
## 01 data ingestion 

goals:\
-load Idaho State Survey and USGS data\
-identify and filter to data relevant to gold mining and prospectivity\
-export gold data subset for further analysis\
\
-load USGS National Geochemical Database (NGDB) data for Idaho\
-interrogate and convert to spaital analysis-ready geodataframes


### Part 1. import packages and identify local directory
a. import packages needed for data loading and analysis

In [1]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import glob
import os

data_path = "/Users/adbyerly/prospectivity_model/data/"

b. metadata dataframe for sources\
-create dataframe\
-add source URLs\
-add notes about cleaning/processing & update with further work

In [2]:
#metadata

sources = pd.DataFrame({
    'source' : [
        'Idaho Geologic Survey',
        'USGS',
        'USGS',
        'USGS',
        'USGS'
    ],
    
    'table_name' : [
        'mines',
        'ngdb',
        'mrds',
        'magnetic',
        'gravity'
    ],

    'source_url' : [
        'https://www.idahogeology.org/product/dd-1',
        'https://www.usgs.gov/centers/gggsc/science/national-geochemical-database',
        'https://mrdata.usgs.gov/mrds/',
        'https://mrdata.usgs.gov/magnetic/',
        'https://mrdata.usgs.gov/gravity/'
    ],

    'donwload_date' : [
        '2026-05-01',
        '2026-05-01',
        '2026-05-06',
        '2026-05-08',
        '2026-05-18'
    ],

    'original_datum' : [
        'WGS84',
        'Mixed(WGS84, NAD83, NAD27, NaN',
        '',
        'NAD27',
        'NAD27'
    ],

    'notes' : [
        None,
        'projected unknonwn datum values to NAD27 per metadata',
        None,
        'NAD27 per metadata',
        'NAD27 per metadata'
    ]})

sources.head()

,source,table_name,source_url,donwload_date,original_datum,notes
0,Idaho Geologic Survey,mines,https://www.idahogeology.org/product/dd-1,2026-05-01,WGS84,NaN
1,USGS,ngdb,https://www.usgs.gov/centers/gggsc/science/nat...,2026-05-01,"Mixed(WGS84, NAD83, NAD27, NaN",projected unknonwn datum values to NAD27 per m...
2,USGS,mrds,https://mrdata.usgs.gov/mrds/,2026-05-06,,NaN
3,USGS,magnetic,https://mrdata.usgs.gov/magnetic/,2026-05-08,NAD27,NAD27 per metadata
4,USGS,gravity,https://mrdata.usgs.gov/gravity/,2026-05-18,NAD27,NAD27 per metadata


### Part 2. load raw data and process

a. load raw data from Idaho Geologic Survey mines database\
-load data\
-inspect format and identify data relevant to gold\
-filter to gold data\
-export layer file for future analysis

In [3]:
#shapefile import
mines_gdf = gpd.read_file(data_path + "raw/MinesPubDD-1_2023/MinesPubDD-1_Shapefile_2023/Mines_DD-1_2023.shp")
mines_gdf.head()

,IGSID,NameList,CommodityL,CompanyLis,NAD27lon,NAD27lat,DMSLON,DMSLAT,WGS84lon,WGS84lat,...,Undergroun,Surface,Drilled,Producer,Processing,LocationTy,NameWeb,CommodityW,CompanyWeb,geometry
0,AS0001,Thimbleberry Iron Prospect,iron,NaN,-111.380187,44.673231,"111° 22' 51.557"" W","44° 40' 23.376"" N",-111.380988,44.673160,...,0,0,0,0,0,0,|Thimbleberry Iron Prospect|,|iron|,NaN,POINT (-111.38099 44.67316)
1,AS0002,Tick Heaven Talc Prospect,talc,NaN,-111.368778,44.667711,"111° 22' 10.480"" W","44° 40' 3.504"" N",-111.369578,44.667640,...,0,0,0,0,0,1,|Tick Heaven Talc Prospect|,|talc|,NaN,POINT (-111.36958 44.66764)
2,AS0003,Idaho Driftwood,stone,NaN,-111.503775,44.643262,"111° 30' 16.493"" W","44° 38' 35.482"" N",-111.504581,44.643189,...,0,0,0,0,0,1,|Idaho Driftwood|,|stone|,NaN,POINT (-111.50458 44.64319)
3,AS0004,Idaho Driftwood #1,stone,NaN,-111.504885,44.638542,"111° 30' 20.489"" W","44° 38' 18.490"" N",-111.505691,44.638469,...,0,0,0,0,0,1,|Idaho Driftwood #1|,|stone|,NaN,POINT (-111.50569 44.63847)
4,AS0005,Unnamed location,limestone,NaN,-111.425486,44.593033,"111° 25' 34.638"" W","44° 35' 34.657"" N",-111.426288,44.592960,...,0,0,0,0,0,0,|Unnamed location|,|limestone|,NaN,POINT (-111.42629 44.59296)


In [4]:
mines_gdf.shape

(9413, 35)

In [5]:
mines_gdf.columns

Index(['IGSID', 'NameList', 'CommodityL', 'CompanyLis', 'NAD27lon', 'NAD27lat',
       'DMSLON', 'DMSLAT', 'WGS84lon', 'WGS84lat', 'q24k', 'q100k', 'q250k',
       'plss', 'plssTOWNSH', 'plssRANGE', 'plssSECTIO', 'plssQQSECT',
       'plssQSECTI', 'CountyName', 'SurfaceMan', 'OrangeNumb', 'ZIP',
       'MiningDist', 'Placer', 'Undergroun', 'Surface', 'Drilled', 'Producer',
       'Processing', 'LocationTy', 'NameWeb', 'CommodityW', 'CompanyWeb',
       'geometry'],
      dtype='str')

In [6]:
mines_gdf[mines_gdf["CommodityL"].str.contains("gold", case=False, na=False)]["CommodityL"].value_counts()

CommodityL
gold                                                  1004
gold, silver                                           593
silver, gold                                           503
gold, black sand                                       105
gold, rare-earths, zirconium, black sand, monazite      91
                                                      ... 
silver, lead, copper, cobalt, gold                       1
copper, quartz, zinc, gold, lead                         1
abrasives, gold, garnet, black sand                      1
gold, copper, zinc                                       1
gold, tellurium, silver, mercury, lead                   1
Name: count, Length: 1027, dtype: int64

In [7]:
#create a subset of this dataframe that only includes data relevant to gold
gold_gdf = mines_gdf[mines_gdf["CommodityL"].str.contains("gold", case=False, na=False)]
gold_gdf.head()

,IGSID,NameList,CommodityL,CompanyLis,NAD27lon,NAD27lat,DMSLON,DMSLAT,WGS84lon,WGS84lat,...,Undergroun,Surface,Drilled,Producer,Processing,LocationTy,NameWeb,CommodityW,CompanyWeb,geometry
16,AS0017,"Kilgore Project, Blue Ledge Mining Company, Un...",gold,"Kilgore Minerals Ltd., Otis Gold Corporation, ...",-111.994597,44.436048,"111° 59' 43.522"" W","44° 26' 9.484"" N",-111.995423,44.435968,...,0,0,1,0,0,1,|Kilgore Project|Blue Ledge Mining Company|Unn...,|gold|,|Kilgore Minerals Ltd.|Otis Gold Corporation|P...,POINT (-111.99542 44.43597)
17,AS0018,"Blue Ledge Mining Company, Kilgore project",gold,"Kennecott, Placer Dome, Kilgore Minerals Ltd.,...",-111.992097,44.433548,"111° 59' 34.522"" W","44° 26' 0.485"" N",-111.992923,44.433468,...,1,0,1,0,0,1,|Blue Ledge Mining Company|Kilgore project|,|gold|,|Kennecott|Placer Dome|Kilgore Minerals Ltd.|O...,POINT (-111.99292 44.43347)
30,BA0001,Indian Creek,"gold, copper",NaN,-116.767347,45.048548,"116° 46' 6.045"" W","45° 2' 54.353"" N",-116.768346,45.048431,...,0,0,0,0,0,1,|Indian Creek|,|gold|copper|,NaN,POINT (-116.76835 45.04843)
33,BA0004,Rebound,"copper, gold, zinc, lead, silver",NaN,-116.624544,44.900236,"116° 37' 31.943"" W","44° 54' 0.444"" N",-116.625540,44.900123,...,0,0,0,0,0,0,|Rebound|,|copper|gold|zinc|lead|silver|,NaN,POINT (-116.62554 44.90012)
35,BA0006,North Hornet Mine,"uranium, gold, tantalum, niobium, abrasives, c...",NaN,-116.638173,44.890766,"116° 38' 21.012"" W","44° 53' 26.352"" N",-116.639170,44.890653,...,1,0,0,1,1,1,|North Hornet Mine|,|uranium|gold|tantalum|niobium|abrasives|ceriu...,NaN,POINT (-116.63917 44.89065)


b. load raw geochemical data from the "USGS National Geochemical Database (NGDB)"\
-data filtered to the state of Idaho before bulk download from USGS website\
-load data\
-interrogate data and datums\
-standaradize CRS

In [8]:
# csv - load all geochemical files from the downloaded folder into a dictionary
files = glob.glob(data_path + "raw/ngdbrock-fUS16/*.txt")
ngdb_dfs={}

for file in files:
    name = os.path.basename(file).replace(".txt", "")
    df = pd.read_csv(file, low_memory=False)
    ngdb_dfs[name] = df

ngdb_dfs.keys()

dict_keys(['tblRockGeoData', 'xtbOtherChem', 'xtbEsChem', 'xtbUnknownChem', 'xtbIcpmsChem', 'xtbIcpaesChem', 'xtbMajorChem', 'xtbXrfChem', 'xtbNaaChem'])

In [9]:
# rename dictionary items to more straightforward names based on metadata here: https://mrdata.usgs.gov/metadata/ngdbrock.html
renaming = {"tblRockGeoData" : "Rock_Data", #rock sample data
            "xtbOtherChem" : "Other_chem", #atomic absorption spectrometry, colorimetry, ion selective electrode (except Cl & F), fluorometry, or fire assay
            "xtbEsChem" : "ES", #electron spectroscopy
            "xtbUnknownChem" : "Unknown_chem", #analyses derived from undocumented methods
            "xtbIcpmsChem" : "ICPMS", #inductively coupled plasma mass spectrometry
            "xtbIcpaesChem" : "ICPAES", #inductively coupled plasma-atomic emission spectrometry
            "xtbMajorChem" : "Major_element", #major element oxide analyses plus forms-of-carbon, forms-of-sulfur, chlorine, and fluorine data.
            "xtbXrfChem" : "XRF", #x-ray fluorescence
            "xtbNaaChem" : "NAA" #instrumental neutron activation analysis or delayed neutron activation analysis
           }

ngdb = {renaming.get(k, k): v for k, v in ngdb_dfs.items()} 
ngdb.keys()

dict_keys(['Rock_Data', 'Other_chem', 'ES', 'Unknown_chem', 'ICPMS', 'ICPAES', 'Major_element', 'XRF', 'NAA'])

In [10]:
# examine dataframes; explanation of tables and features is here: https://mrdata.usgs.gov/ngdb/rock/about.php
# primary key for samples is "LAB_ID", georeferenced in 'Rock_Data' table and used as a foreign key in all other tables

for name, df in ngdb.items():
    print(name)
    print(df.head())
    print(df.columns)
    print(df.shape)
    print("-" * 25)

Rock_Data
   lab_id job_id         submitter    date_sub field_id state        country  \
0  AAV161   HM42  Gott, Garland B.  19660721.0    CD008    ID  United States   
1  AAV162   HM42  Gott, Garland B.  19660721.0   CD008A    ID  United States   
2  AAV163   HM42  Gott, Garland B.  19660721.0    CD009    ID  United States   
3  AAV164   HM42  Gott, Garland B.  19660721.0   CD009A    ID  United States   
4  AAV165   HM42  Gott, Garland B.  19660721.0    CD010    ID  United States   

  datum spheroid  latitude  ...  mineralztn alteration struct_src dep_envirn  \
0   NaN      NaN  47.47778  ...         NaN        NaN        NaN        NaN   
1   NaN      NaN  47.47778  ...         NaN        NaN        NaN        NaN   
2   NaN      NaN  47.47667  ...         NaN        NaN        NaN        NaN   
3   NaN      NaN  47.47667  ...         NaN        NaN        NaN        NaN   
4   NaN      NaN  47.47556  ...         NaN        NaN        NaN        NaN   

  source_rk metamrphsm facie

In [11]:
# process NGDB data

# 'Rock_Data'
# first we need to know the datum for lat and long values; 
# from the metadata, legacy data are North American Datum of 1927 on the Clarke 1866 ellipsoid (https://mrdata.usgs.gov/metadata/ngdbrock.faq.html)

print(ngdb['Rock_Data']['datum'].unique())
print(ngdb['Rock_Data']['datum'].value_counts(dropna=False))


# --- UNCERTAINTY  --- 
# -------------------- it is unclear if ALL of the data are NAD27; there are a large amount of NaN values
# -------------------- assume that these are all NAD27 for regional analysis 
# -------------------- investigate further and test alternate scenarios for detailed work
# --- UNCERTAINTY  --- 


# investigate vintage of samples with undefined datum
rock = ngdb['Rock_Data']

dates = pd.to_datetime(
    pd.to_numeric(
        rock.loc[rock['datum'].isna(), 'date_sub'], errors = 'coerce'),
    format = '%Y%m%d', errors = 'coerce')

dates.agg(['min', 'max'])


<StringArray>
[nan, 'WGS84', 'NAD83', 'NAD27']
Length: 4, dtype: str
datum
NaN      26152
WGS84      153
NAD27      115
NAD83      101
Name: count, dtype: int64


min   1964-04-07
max   2001-02-23
Name: date_sub, dtype: datetime64[us]

In [12]:
# un-datumed samples range in submission date from 1964 to 2001
# assume these are NAD 1927 per metadata but note uncertainty highlighted above

rock['datum'] = rock['datum'].fillna('NAD27')
rock.head()


,lab_id,job_id,submitter,date_sub,field_id,state,country,datum,spheroid,latitude,...,mineralztn,alteration,struct_src,dep_envirn,source_rk,metamrphsm,facies_grd,prep,mesh_size,Unnamed: 31
0,AAV161,HM42,"Gott, Garland B.",19660721.0,CD008,ID,United States,NAD27,NaN,47.47778,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AAV162,HM42,"Gott, Garland B.",19660721.0,CD008A,ID,United States,NAD27,NaN,47.47778,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AAV163,HM42,"Gott, Garland B.",19660721.0,CD009,ID,United States,NAD27,NaN,47.47667,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AAV164,HM42,"Gott, Garland B.",19660721.0,CD009A,ID,United States,NAD27,NaN,47.47667,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AAV165,HM42,"Gott, Garland B.",19660721.0,CD010,ID,United States,NAD27,NaN,47.47556,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# create geopandas dataframes for sample table ('Rock_Data'), split by datum

epsg = {
    'NAD27' : 4267,
    'WGS84' : 4326,
    'NAD83' : 4269
}

# dicitonary of 'Rock_Data' split into geodataframe by datum (3 datums)
# these dataframes won't be used for analysis, but serve as a record of and are ready for subsequent reprojection

rock_gdfs = {}

for d, df in rock.groupby('datum'):
    rock_gdfs[d] = gpd.GeoDataFrame(
        df.copy(),
        geometry = gpd.points_from_xy(df['longitude'], df['latitude']),
        crs = f"EPSG:{epsg[d]}")

In [14]:
# project geodataframes in dictionary to common CRS (UTM Zone 11N - not perfect but a good compromise))
target_crs = 'EPSG:32611'

rock_gdfs_UTM = {key: gdf.to_crs(target_crs) for key, gdf in rock_gdfs.items()}

# concatenate reprojected geodataframes into one dataframe
rock_UTM = pd.concat(rock_gdfs_UTM.values(), ignore_index=True)

# preserve origional datum and add column to reference new datum
rock_UTM.rename(columns={'datum':'original_datum'}, inplace = True)
rock_UTM['projected_datum'] = 'UTM'

print(rock_UTM.head())
print(rock_UTM.shape)

   lab_id job_id         submitter    date_sub field_id state        country  \
0  AAV161   HM42  Gott, Garland B.  19660721.0    CD008    ID  United States   
1  AAV162   HM42  Gott, Garland B.  19660721.0   CD008A    ID  United States   
2  AAV163   HM42  Gott, Garland B.  19660721.0    CD009    ID  United States   
3  AAV164   HM42  Gott, Garland B.  19660721.0   CD009A    ID  United States   
4  AAV165   HM42  Gott, Garland B.  19660721.0    CD010    ID  United States   

  original_datum spheroid  latitude  ...  struct_src dep_envirn source_rk  \
0          NAD27      NaN  47.47778  ...         NaN        NaN       NaN   
1          NAD27      NaN  47.47778  ...         NaN        NaN       NaN   
2          NAD27      NaN  47.47667  ...         NaN        NaN       NaN   
3          NAD27      NaN  47.47667  ...         NaN        NaN       NaN   
4          NAD27      NaN  47.47556  ...         NaN        NaN       NaN   

  metamrphsm facies_grd prep mesh_size Unnamed: 31  \
0 

In [15]:
# standardized dictionary with base table converted to one CRS ready for analysis
ngdb_UTM = {k: v.copy() for k, v in ngdb.items()}
ngdb_UTM['Rock_Data'] = rock_UTM.copy()
ngdb_UTM.keys()

dict_keys(['Rock_Data', 'Other_chem', 'ES', 'Unknown_chem', 'ICPMS', 'ICPAES', 'Major_element', 'XRF', 'NAA'])

In [16]:
ngdb_UTM['Rock_Data'].head()

,lab_id,job_id,submitter,date_sub,field_id,state,country,original_datum,spheroid,latitude,...,struct_src,dep_envirn,source_rk,metamrphsm,facies_grd,prep,mesh_size,Unnamed: 31,geometry,projected_datum
0,AAV161,HM42,"Gott, Garland B.",19660721.0,CD008,ID,United States,NAD27,NaN,47.47778,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (590358.648 5258941.432),UTM
1,AAV162,HM42,"Gott, Garland B.",19660721.0,CD008A,ID,United States,NAD27,NaN,47.47778,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (590358.648 5258941.432),UTM
2,AAV163,HM42,"Gott, Garland B.",19660721.0,CD009,ID,United States,NAD27,NaN,47.47667,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (590255.819 5258816.458),UTM
3,AAV164,HM42,"Gott, Garland B.",19660721.0,CD009A,ID,United States,NAD27,NaN,47.47667,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (590255.819 5258816.458),UTM
4,AAV165,HM42,"Gott, Garland B.",19660721.0,CD010,ID,United States,NAD27,NaN,47.47556,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (590131.888 5258691.16),UTM


c. load data from the USGS Mineral Resource Data System (MRDS)\
-data filtered to the state of Idaho before bulk download from USGS website\
-load data\
-interrogate data and datums\
-convert to GeoDataFrame

In [17]:
# CSV import
mrds = pd.read_csv(data_path + 'raw/MRDS/mrds-fUS16.txt', low_memory=False)

In [18]:
print(mrds.head())
print(mrds.columns)
print(mrds.shape)

     dep_id                                                url  mrds_id  \
0  10008100  https://mrdata.usgs.gov/mrds/show-mrds.php?dep...  D000011   
1  10008372  https://mrdata.usgs.gov/mrds/show-mrds.php?dep...  D000470   
2  10008373  https://mrdata.usgs.gov/mrds/show-mrds.php?dep...  D000472   
3  10008375  https://mrdata.usgs.gov/mrds/show-mrds.php?dep...  D000474   
4  10008376  https://mrdata.usgs.gov/mrds/show-mrds.php?dep...  D000477   

  mas_id                site_name  latitude  longitude  region        country  \
0    NaN            Buckskin Mine  43.98321 -115.88429     NaN  United States   
1    NaN             Little Falls  44.08321 -115.75096     NaN  United States   
2    NaN  Boundary County Deposit  48.88319 -116.83441     NaN  United States   
3    NaN            Boulder Creek  44.05179 -114.56232     NaN  United States   
4    NaN                 Sawtooth  44.03549 -114.97673     NaN  United States   

   state  ...                                                r

In [19]:
mrds['dep_type'].unique()

<StringArray>
[                                                nan,
                                            'Placer',
                          'HYDROTHERMAL BEDDED VEIN',
                                              'Vein',
                                       'Replacement',
                                       'Sedimentary',
                                       'Metamorphic',
                                           'Unknown',
                                     'Stream Placer',
                                             'Float',
 ...
          'SYNGENETIC SEDIMENT-HOSTED (CARBON-RICH)',
                                     'STREAM PLACER',
                             'VEINS AND SHEAR ZONES',
                                   'VEIN/SHEAR ZONE',
                 'VEIN, SHEAR ZONE CHIMNEY OR STOCK',
                                 'VEIN, REPLACEMENT',
                           'STRATIFORM DISSEMINATED',
 'HYDROTHERMAL VEINS, STOCKWORK, CHIMNEYS, BRECCIAS',
         

In [20]:
# convert to GeoDataFrame
mrds_gdf = gpd.GeoDataFrame(
    mrds.copy(), 
    geometry = gpd.points_from_xy (mrds['longitude'], mrds['latitude']),
    crs = 'EPSG:4326'
)

mrds_gdf.head()

,dep_id,url,mrds_id,mas_id,site_name,latitude,longitude,region,country,state,...,yfp_ba,yr_fst_prd,ylp_ba,yr_lst_prd,dy_ba,disc_yr,prod_yrs,discr,score,geometry
0,10008100,https://mrdata.usgs.gov/mrds/show-mrds.php?dep...,D000011,NaN,Buckskin Mine,43.98321,-115.88429,NaN,United States,Idaho,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,D,POINT (-115.88429 43.98321)
1,10008372,https://mrdata.usgs.gov/mrds/show-mrds.php?dep...,D000470,NaN,Little Falls,44.08321,-115.75096,NaN,United States,Idaho,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,D,POINT (-115.75096 44.08321)
2,10008373,https://mrdata.usgs.gov/mrds/show-mrds.php?dep...,D000472,NaN,Boundary County Deposit,48.88319,-116.83441,NaN,United States,Idaho,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,D,POINT (-116.83441 48.88319)
3,10008375,https://mrdata.usgs.gov/mrds/show-mrds.php?dep...,D000474,NaN,Boulder Creek,44.05179,-114.56232,NaN,United States,Idaho,...,NaN,NaN,NaN,NaN,NaN,1919.0,NaN,NaN,C,POINT (-114.56232 44.05179)
4,10008376,https://mrdata.usgs.gov/mrds/show-mrds.php?dep...,D000477,NaN,Sawtooth,44.03549,-114.97673,NaN,United States,Idaho,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,D,POINT (-114.97673 44.03549)


d. load magnetic data from the USGS\
-downloaded data encompass the entire US\
-from metadata, datum is NAD27

In [21]:
# CSV import
mag = pd.read_csv(data_path + 'raw/magnetic/magnetic.xyz', sep = '\s+', header=None, low_memory=False)

In [22]:
print(mag.head())
print(mag.columns)
print(mag.shape)

         0        1      2
0 -94.7670  24.9919 -201.1
1 -94.7475  24.9917 -200.8
2 -94.7279  24.9915 -200.5
3 -94.7083  24.9912 -200.2
4 -94.6887  24.9910 -199.8
Index([0, 1, 2], dtype='int64')
(2581554, 3)


In [23]:
# name columns
mag.columns = ['X', 'Y', 'Z']
print(mag.head())

         X        Y      Z
0 -94.7670  24.9919 -201.1
1 -94.7475  24.9917 -200.8
2 -94.7279  24.9915 -200.5
3 -94.7083  24.9912 -200.2
4 -94.6887  24.9910 -199.8


d. load Bouguer gravity data from the USGS\
-downloaded data econmpassess the entire US
-from metadata, datum is NAD27

In [24]:
# CSV import
grav = pd.read_csv(data_path + 'raw/gravity/bouguer.xyz', sep =  '\s+', header=None, low_memory=False)

In [25]:
print(grav.head())
print(grav.columns)
print(grav. shape)

         0        1     2
0 -97.6570  24.6384  23.0
1 -97.6180  24.6390  25.0
2 -97.5790  24.6396  26.0
3 -97.5400  24.6402  26.0
4 -97.5011  24.6408  27.0
Index([0, 1, 2], dtype='int64')
(728657, 3)


In [26]:
# name columns
grav.columns = ['X', 'Y', 'Z']
print(grav.head())

         X        Y     Z
0 -97.6570  24.6384  23.0
1 -97.6180  24.6390  25.0
2 -97.5790  24.6396  26.0
3 -97.5400  24.6402  26.0
4 -97.5011  24.6408  27.0


### Part 3. export

a. save cleaned Idaho Geological Survey data

In [27]:
gold_gdf.to_file(data_path + "processed/idaho_gold_mines.geojson", driver="GeoJSON")

b. save cleaned USGS National Geochemical Database (NGDB) data reprojected to common CRS

In [28]:
for name, df in ngdb_UTM.items():
    df.to_csv(data_path + "processed/NGDB/" + f"{name}.csv", index=False)

c. save cleaned USGS Mineral Resource Data System (MRDS) data 

In [29]:
mrds_gdf.to_file(data_path + "processed/mrds_gdf.geojson", driver="GeoJSON")

d. save cleaned USGS magnetic survey data

In [30]:
mag.to_csv(data_path + "processed/mag_USGS_NAD27.csv", index=False)

e. save cleaned USGS Bouguer gravity data

In [31]:
grav.to_csv(data_path + "processed/Bouguer_grav_NAD27.csv", index=False)